# UN speeches and conflict transitions

SDG 16. We study whether cooperation and military-threat themes differ across transitions into and out of government-side state-based armed conflict. The planned predictive question is whether rhetoric adds information beyond conflict history and military spending.

This notebook reproduces preparation and descriptive results. Prediction is not implemented yet. Configure `.env` and obtain `outputs/rhetoric_full.sqlite` as described in README.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from scripts.prepare_analysis import prepare
from scripts.review_and_eda import main as reproduce_eda

ROOT = Path.cwd()

## 1. Prepare the country-year table

Speeches are matched to UCDP and SIPRI by canonical country identifiers. The outcome is conflict in exactly the following calendar year, whether or not a speech exists in that year. Missing spending stays missing; 0.03 means 3% of GDP.

The main population uses current UN member-state identities with historical exclusions, not a historical membership roster. UCDP identifies government-side involvement, not necessarily the physical battlefield.

In [ ]:
result = prepare()
main = result.analysis.loc[result.analysis.main_population]
display(pd.Series({'speeches': len(result.analysis), 'main_population': len(main),
                   'countries': main.entity_id.nunique(),
                   'observed_spending': main.milex_share_gdp_t.notna().sum()}))
display(result.audits['coverage_by_year'])

## 2. Load full-speech scores and reproduce EDA

The cached ModernBERT scores cover the entire cleaned speech in non-overlapping chunks of at most 480 tokens. Scores are token-weighted averages. The two themes are evaluated independently. Natural language and negations are retained.

These scores measure compatibility with themes, not sentiment, actual trust, aggression or conflict probability. Generating them from scratch is a separate, expensive step; this notebook does not rerun inference.

In [ ]:
reproduce_eda()
eda = ROOT / 'outputs/whole_speech_eda'
display(pd.read_csv(eda / 'transition_counts.csv'))
display(pd.read_csv(eda / 'transition_feature_summaries.csv'))
display(pd.read_csv(eda / 'country_cluster_bootstrap_differences.csv'))

## 3. Inspect distributions and time trends

The EDA uses speech years 1990–2024 with observed current and next-year conflict. 2025 has no observed next-year target. Session 81 is outside scope.

For uncertainty we resample whole countries, keeping their years together. Intervals are exploratory, not adjusted for multiple comparisons or shared global shocks. Spending comparisons use the same SIPRI-complete rows for all three features.

In [ ]:
for name in ['score_ecdf_by_conflict_transition.png',
             'military_spending_ecdf_by_conflict_transition.png',
             'whole_speech_score_trends_1990_2024.png']:
    display(Image(filename=str(eda / name)))

## Interpretation and next steps

Read `EDA_RESULTS.md` for numerical summaries and limitations. Independently review `outputs/whole_speech_eda/passage_review_blinded.csv` before looking at `PASSAGE_REVIEW.md` or the score key. The existing passage review is AI-assisted, not independent human validation.

Next: compare prevalence, conflict persistence and conflict-history models with models adding spending and then rhetoric, using chronological validation. EDA differences alone do not establish predictive improvement or causality. See `RESEARCH_DESIGN.md`.